# Notebook6 - Forecasting

用同時段 baseline 和 rolling residual 建立 prediction interval，進行 forecasting anomaly 和 capacity early warning。

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])

## Step 1 - 建立 forecast baseline 與 prediction interval

In [ ]:
target = "traffic_total"
rows = []
for port_id, g in features.groupby("port_id", sort=False):
    g = g.sort_values("timestamp").copy()
    g["slot"] = g["timestamp"].dt.dayofweek.astype(str) + "-" + g["timestamp"].dt.hour.astype(str) + "-" + (g["timestamp"].dt.minute // 5).astype(str)
    seasonal = g.groupby("slot")[target].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    moving = g[target].shift(1).rolling(12, min_periods=3).mean()
    g["y_hat"] = seasonal.fillna(moving).fillna(g[target].expanding().mean())
    residual = (g[target] - g["y_hat"]).abs()
    band = residual.shift(1).rolling(24, min_periods=6).quantile(0.95).fillna(residual.quantile(0.95))
    g["y_hat_lower"] = (g["y_hat"] - band).clip(lower=0)
    g["y_hat_upper"] = g["y_hat"] + band
    capacity = g[target].quantile(0.995) * 1.15
    g["capacity"] = capacity
    g["forecast_positive_anomaly"] = g[target] > g["y_hat_upper"]
    g["forecast_negative_anomaly"] = g[target] < g["y_hat_lower"]
    g["forecast_30m"] = g["y_hat"].shift(-6)
    g["early_warning_30m"] = g["forecast_30m"] > capacity * 0.80
    rows.append(g)

forecast = pd.concat(rows, ignore_index=True)
display(forecast[["timestamp", "port_id", target, "y_hat", "y_hat_lower", "y_hat_upper", "early_warning_30m"]].head())

## Step 2 - 視覺化：Actual vs Forecast + prediction interval

In [ ]:
port_id = "port-id7431"
p = forecast[forecast["port_id"] == port_id].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(p["timestamp"], p[target], label="actual", linewidth=1)
ax.plot(p["timestamp"], p["y_hat"], label="forecast", linewidth=1.2)
ax.fill_between(p["timestamp"], p["y_hat_lower"], p["y_hat_upper"], alpha=0.18, label="prediction interval")
ax.plot(p["timestamp"], p["capacity"], label="capacity", color="tab:red", linestyle="--")
warnings_df = p[p["early_warning_30m"]]
ax.scatter(warnings_df["timestamp"], warnings_df[target], color="tab:orange", s=18, label="early warning")
for _, event in event_catalog.iterrows():
    if event["port_id"] in [port_id, "MULTI"]:
        ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
ax.set_title(f"{port_id} - Forecasting and capacity early warning")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

## Step 3 - 視覺化：Forecasting anomaly summary

In [ ]:
summary = (
    forecast.groupby("event_label")
    .agg(
        rows=("timestamp", "size"),
        positive_anomaly=("forecast_positive_anomaly", "sum"),
        negative_anomaly=("forecast_negative_anomaly", "sum"),
        early_warning=("early_warning_30m", "sum"),
    )
)
summary["positive_rate"] = summary["positive_anomaly"] / summary["rows"]
display(summary.sort_values("positive_rate", ascending=False))

fig, ax = plt.subplots(figsize=(12, 5))
summary["early_warning"].sort_values().plot(kind="barh", ax=ax, color="tab:orange")
ax.set_title("Early warning count by event label")
ax.set_xlabel("count")
plt.tight_layout()
show_fig(fig)

In [ ]:
keep = [
    "timestamp", "device_id", "port_id", "port_role", "event_label", "event_id",
    target, "y_hat_lower", "y_hat", "y_hat_upper", "capacity",
    "forecast_positive_anomaly", "forecast_negative_anomaly", "forecast_30m", "early_warning_30m",
]
out_path = DATA_PROCESSED / "forecast_results.csv"
forecast[keep].to_csv(out_path, index=False)
print(f"saved: {out_path}")